<a href="https://colab.research.google.com/github/ProgramNewbbie/FPGA-Transformer-Accelerator-Extension/blob/main/Extension_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time                 # 시간 측정 및 sleep() 함수 사용
import threading            # 멀티스레드 생성 및 실행
import queue                # 스레드 간 데이터를 안전하게 전달하는 Queue

free_buffer_queue = queue.Queue()      # 현재 사용 가능한 버퍼 번호를 저장하는 Queue
free_buffer_queue.put(0)               # 버퍼 0을 사용 가능 상태로 등록
free_buffer_queue.put(1)               # 버퍼 1을 사용 가능 상태로 등록

fpga_task_queue = queue.Queue()        # FPGA에 전달할 작업을 저장하는 Queue
fpga_done_queue = queue.Queue()        # FPGA 연산 완료를 전달하는 Queue
cpu_post_queue = queue.Queue()         # CPU 후처리 작업을 전달하는 Queue

input_buffers = {0: None, 1: None}     # ❌ 시뮬레이션용 입력 버퍼
output_buffers = {0: None, 1: None}    # ❌ 시뮬레이션용 출력 버퍼

# ✅ from pynq import allocate, Overlay                  # PYNQ 라이브러리 사용
# ✅ overlay = Overlay("/home/xilinx/design.bit")        # FPGA Bitstream 다운로드
# ✅ fpga_ip = overlay.mmult_accel_0                     # FPGA IP 객체 생성
# ✅ input_buffers  = { 0: allocate(shape=(64,768), dtype='int8'),   # 입력 버퍼를 CMA 메모리에 할당
# ✅                    1: allocate(shape=(64,768), dtype='int8') }
# ✅ output_buffers = { 0: allocate(shape=(64,768), dtype='int8'),   # 출력 버퍼를 CMA 메모리에 할당
# ✅                    1: allocate(shape=(64,768), dtype='int8') }

def mock_fpga_kernel(buf_idx):          # ❌ FPGA 동작을 흉내 내는 시뮬레이션 함수
    in_data = input_buffers[buf_idx]    # 입력 버퍼 데이터 읽기
    time.sleep(0.5)                     # FPGA 계산 시간을 0.5초로 가정
    output_buffers[buf_idx] = f"[{in_data}의 H/W 연산 완료]"   # 결과를 출력 버퍼에 저장

# ✅ def real_fpga_kernel(buf_idx):
# ✅     input_buffers[buf_idx].flush()                         # CPU Cache → DDR 동기화
# ✅     fpga_ip.write(0x10, input_buffers[buf_idx].physical_address)   # 입력 버퍼 주소 전달
# ✅     fpga_ip.write(0x1C, output_buffers[buf_idx].physical_address)  # 출력 버퍼 주소 전달
# ✅     fpga_ip.write(0x00, 0x01)                              # FPGA 연산 시작

def fpga_async_worker():                # FPGA 작업을 수행하는 스레드
    while True:                         # 종료 신호가 올 때까지 반복
        buf_idx = fpga_task_queue.get() # FPGA 작업 Queue에서 버퍼 번호 가져오기
        if buf_idx is None: break       # None이면 스레드 종료
        mock_fpga_kernel(buf_idx)       # ❌ → ✅ 실제 구현 시 real_fpga_kernel(buf_idx)
        fpga_done_queue.put(buf_idx)    # FPGA 연산 완료를 완료 Queue에 등록
        fpga_task_queue.task_done()     # 현재 작업이 끝났음을 Queue에 알림

def interrupt_worker():                 # ❌ 시뮬레이션용 인터럽트 처리 함수
    while True:                         # 종료 신호가 올 때까지 반복
        buf_idx = fpga_done_queue.get() # 완료된 버퍼 번호 가져오기
        if buf_idx is None: break       # None이면 스레드 종료
        cpu_post_queue.put(buf_idx)     # CPU 후처리 Queue로 전달
        fpga_done_queue.task_done()     # 완료 Queue 작업 종료 처리

# ✅ my_interrupt = overlay.mmult_accel_0.interrupt      # FPGA 인터럽트 객체 생성
# ✅ def real_interrupt_worker():
# ✅     while True:
# ✅         buf_idx = fpga_done_queue.get()             # 완료된 버퍼 번호 가져오기
# ✅         if buf_idx is None: break                   # 종료 신호 확인
# ✅         my_interrupt.wait()                         # FPGA 인터럽트 발생 대기
# ✅         my_interrupt.clear()                        # 인터럽트 플래그 초기화
# ✅         output_buffers[buf_idx].invalidate()        # DDR → CPU Cache 동기화
# ✅         cpu_post_queue.put(buf_idx)                 # CPU 후처리 Queue 전달
# ✅         fpga_done_queue.task_done()                 # 완료 Queue 작업 종료 처리

def run_simulation(total_data_count):   # 전체 데이터 처리 과정 실행
    start_time = time.time()            # 실행 시작 시간 저장

    t1 = threading.Thread(target=fpga_async_worker, daemon=True)    # FPGA 작업 스레드 생성
    t2 = threading.Thread(target=interrupt_worker, daemon=True)     # 인터럽트 처리 스레드 생성
                                                                    # ❌ → ✅ real_interrupt_worker
    t1.start()                          # FPGA 스레드 실행
    t2.start()                          # 인터럽트 스레드 실행

    for i in range(1, total_data_count + 1):   # 입력 데이터 개수만큼 반복
        buf_idx = None                  # 사용할 버퍼 번호 초기화
        is_waiting = False              # 버퍼 대기 여부 표시

        while buf_idx is None:          # 사용할 버퍼를 얻을 때까지 반복
            while not cpu_post_queue.empty():   # 후처리할 결과가 존재하면
                done_idx = cpu_post_queue.get() # 완료된 버퍼 번호 가져오기
                result_data = output_buffers[done_idx] # 출력 데이터 읽기
                print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Output {done_idx} '{result_data}' 후처리 완료")
                free_buffer_queue.put(done_idx) # 사용이 끝난 버퍼를 반환
                is_waiting = False              # 대기 상태 해제

            try:
                buf_idx = free_buffer_queue.get_nowait() # 사용 가능한 버퍼 가져오기
            except queue.Empty:                          # 사용 가능한 버퍼가 없으면
                if not is_waiting:
                    is_waiting = True                    # 대기 상태 표시
                time.sleep(0.01)                         # Busy Waiting 방지

        input_buffers[buf_idx] = f"데이터 {i}"          # ❌ 문자열 저장
                                                        # ✅ 실제 구현 시 입력 데이터를 버퍼에 복사
        print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Input {buf_idx}에 데이터 {i} 입력 -> FPGA 전달")
        fpga_task_queue.put(buf_idx)                    # FPGA 작업 Queue에 버퍼 등록

    while free_buffer_queue.qsize() < 2:               # 두 버퍼가 모두 반환될 때까지 반복
        if not cpu_post_queue.empty():                 # 아직 후처리할 결과가 있으면
            done_idx = cpu_post_queue.get()            # 완료된 버퍼 번호 가져오기
            print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Output {done_idx} 최종 후처리 완료")
            free_buffer_queue.put(done_idx)            # 버퍼 반환
        time.sleep(0.01)                               # CPU 사용률 감소를 위해 잠시 대기

    fpga_task_queue.put(None)                          # FPGA 스레드 종료 신호
    fpga_done_queue.put(None)                          # 인터럽트 스레드 종료 신호
    t1.join()                                          # FPGA 스레드 종료 대기
    t2.join()                                          # 인터럽트 스레드 종료 대기

if __name__ == "__main__":                             # 현재 파일을 직접 실행한 경우
    run_simulation(4)                                  # 총 4개의 데이터를 처리하는 시뮬레이션 실행

[t=0.00s] 🧑‍💻 [Stage 1] Input 0에 데이터 1 입력 -> FPGA 전달
[t=0.00s] 🧑‍💻 [Stage 1] Input 1에 데이터 2 입력 -> FPGA 전달
[t=0.51s] 🧑‍💻 [Stage 1] Output 0 '[데이터 1의 H/W 연산 완료]' 후처리 완료
[t=0.51s] 🧑‍💻 [Stage 1] Input 0에 데이터 3 입력 -> FPGA 전달
[t=1.00s] 🧑‍💻 [Stage 1] Output 1 '[데이터 2의 H/W 연산 완료]' 후처리 완료
[t=1.00s] 🧑‍💻 [Stage 1] Input 1에 데이터 4 입력 -> FPGA 전달
[t=1.51s] 🧑‍💻 [Stage 1] Output 0 최종 후처리 완료
[t=2.01s] 🧑‍💻 [Stage 1] Output 1 최종 후처리 완료
